# FRC Match Team Metrics
Displays the historical metrics of teams at an event.

## Setup
In your virtual environment, install pandas and matplotlib: 
  `pip install pandas matplotlib`
* If you are using VS Code, it should ask you to install the IPython extensions.
* If this next cell runs with no errors, you are all set.

In [50]:
import util
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

## Functions to get Team and Event names
Some parts of this notebook depend on a team you are interested in.  Other parts of this notebook need to look up the team name.  This section defines a function to look up the team name.  Same things for event name.

In [51]:
def get_team_name(team_id):
    url = f'https://www.thebluealliance.com/api/v3/team/{team_id}'
    return util.call_tba_api(url).json()['nickname']

def get_event_name(event_id):
    url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/simple'
    resp = util.call_tba_api(url).json()
    return str(resp['year']) + ' ' + resp['name']

## Default Team and Event

In [52]:
YEAR = '2025'
TEAM = 'frc6223'
EVENT_KEY = '2025wimu'
# EVENT_KEY = '2025wimi'

TEAM_NAME = get_team_name(TEAM)
EVENT_NAME = get_event_name(EVENT_KEY)

print(TEAM_NAME, 'at', EVENT_NAME)

Arsenal of Engineering at 2025 Phantom Lakes Regional


## Teams at the Event

In [53]:
url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/teams/simple'
resp = util.call_tba_api(url).json()
df = pd.DataFrame.from_dict(resp, orient='columns')
print('Team count: ', str(len(df)))
del url, resp
df.drop(columns=['country', 'name', 'team_number'], inplace=True)
df


Team count:  38


,city,key,nickname,state_prov
0,Menomonie,frc10264,STORM,Wisconsin
1,Oregon,frc10553,Orange Overdrive,Wisconsin
2,Hartford,frc1091,Oriole Assault,Wisconsin
3,Milwaukee,frc1220,Rockhoppers - Hilltopper Robotics,Wisconsin
4,Milwaukee,frc1714,MORE Robotics,Wisconsin
5,Milwaukee,frc1732,Hilltopper Robotics,Wisconsin
6,Oak Creek,frc1792,Round Table Robotics,Wisconsin
7,Waukesha,frc2062,CORE 2062,Wisconsin
8,Wales,frc2077,Laser Robotics,Wisconsin
9,Brookfield,frc2202,BEAST Robotics,Wisconsin


## Loop through teams to get metrics

In [54]:
df2 = pd.DataFrame(columns=['key', 'events', 'qual_wins', 'qual_loss', 'qual_ties', 'poff_wins', 'poff_loss', 'poff_ties', 'ranking'])
print('Researching:', end="")
for team in df['key']:
    print('.', end="")
    
    matches = 0
    qual_wins = 0
    qual_loss = 0
    qual_ties = 0
    poff_wins = 0
    poff_loss = 0
    poff_ties = 0
    ranking = []
    
    
    url = f'https://www.thebluealliance.com/api/v3/team/{team}/events/{YEAR}/statuses'
    resp = util.call_tba_api(url).json()
    
    for match in resp:
        details = resp[match]
        
        if details != None and details['qual'] != None:
            matches += 1
            qual_wins += details['qual']['ranking']['record']['wins']
            qual_loss += details['qual']['ranking']['record']['losses']
            qual_ties += details['qual']['ranking']['record']['ties']
            if details['playoff'] != None:
                poff_wins += details['playoff']['record']['wins'] 
                poff_loss += details['playoff']['record']['losses'] 
                poff_ties += details['playoff']['record']['ties'] 
            ranking.append(details['qual']['ranking']['rank'])

    df2.loc[len(df2)] = [team, matches, qual_wins, qual_loss, qual_ties, poff_wins, poff_loss, poff_ties, 0 if len(ranking) == 0 else sum(ranking) / len(ranking)]
    # print('  ', matches, qual_wins, qual_loss, qual_ties, poff_wins, poff_loss, poff_ties, ranking)
    
df = df.join(df2.set_index('key'), on='key')
del df2

Researching:......................................

In [56]:
df.sort_values(by=['qual_wins', 'ranking'], ascending=[False, True], inplace=True, ignore_index=True)
df

,city,key,nickname,state_prov,events,qual_wins,qual_loss,qual_ties,poff_wins,poff_loss,poff_ties,ranking
0,Monterrey,frc4635,PrepaTec - Botbusters,Nuevo León,1,9,1,0,5,0,0,1.0
1,Milwaukee,frc1732,Hilltopper Robotics,Wisconsin,1,8,1,0,3,2,0,3.0
2,Becker,frc4607,C.I.S.,Minnesota,1,7,2,0,0,2,0,7.0
3,Rockford,frc2290,FLYT,Illinois,1,7,2,0,1,2,0,9.0
4,Whitewater,frc6574,Ferradermis,Wisconsin,1,6,3,0,5,1,0,13.0
5,Milwaukee,frc1714,MORE Robotics,Wisconsin,1,6,3,0,0,2,0,22.0
6,Rockford,frc4693,Talk Nerdy to Me,Minnesota,1,5,4,0,0,0,0,27.0
7,Sheboygan,frc6381,Red Raider Robotics,Wisconsin,1,4,5,0,0,0,0,39.0
8,Freedom,frc6318,FE Freedom Engineers,Wisconsin,1,3,6,0,0,0,0,43.0
9,Muskego,frc6421,WarriorBots,Wisconsin,1,3,6,0,0,0,0,47.0


In [57]:
print(df)

               city       key                           nickname   state_prov  \
0         Monterrey   frc4635              PrepaTec - Botbusters   Nuevo León   
1         Milwaukee   frc1732                Hilltopper Robotics    Wisconsin   
2            Becker   frc4607                             C.I.S.    Minnesota   
3          Rockford   frc2290                               FLYT     Illinois   
4        Whitewater   frc6574                        Ferradermis    Wisconsin   
5         Milwaukee   frc1714                      MORE Robotics    Wisconsin   
6         Rockford    frc4693                   Talk Nerdy to Me    Minnesota   
7         Sheboygan   frc6381                Red Raider Robotics    Wisconsin   
8           Freedom   frc6318               FE Freedom Engineers    Wisconsin   
9           Muskego   frc6421                        WarriorBots    Wisconsin   
10        Menomonie  frc10264                              STORM    Wisconsin   
11           Oregon  frc1055

In [58]:
url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/matches/simple'
resp = util.call_tba_api(url).json()
for match in resp:
    if match['match_number'] in [7, 15, 22, 30, 45, 54, 62, 68, 79, 89]:
        print('Match', match['match_number'])
        print('  Red:')
        for team in match['alliances']['red']['team_keys']:
            print('    ', team, df[df['key'] == team].index[0] + 1) 
        print('  Blue:')
        for team in match['alliances']['blue']['team_keys']:
            print('    ', team, df[df['key'] == team].index[0] + 1) 
    # print(match['key'], match['alliances']['red']['team_keys'], match['alliances']['blue']['team_keys'], match['winning_alliance'])